In [68]:
from src.utils import setup_client
import requests
import numpy as np
import pandas as pd
import re
import math

In [92]:
hf_client = setup_client()

✅ Connected to TUDelft LLM proxy


In [70]:
def query_model(prompt, model_name, max_tokens, client):
    """
    Parameters
    ----------
    prompt     : str   — the input text
    model_name : str   — "gpt2", "llama-1b", or "llama-8b"
    max_tokens : int   — how many tokens to generate (capped at 200 by proxy)
    client     : str   — the proxy URL returned by setup_client()

    Returns
    -------
    dict with keys:
        "answer"      : str   — the generated text
        "logprobs"    : dict  — {token: logprob}
        "token_probs" : dict  — {token: probability}
    """
    response = requests.post(
        f"{client}/generate",
        json={
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("⏳ Rate limit hit — wait a moment and try again.")
        return None

    if response.status_code == 400:
        print(f"❌ Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

## Exercise 1 - Generate **Open Question** test sets
For the exercise below, we will need open questions instead of MCQs. We ask you to generate two question sets with each 5 questions. The questions must range from easy to difficult questions for an LLM to answer. The first question set you must generate yourself. Think carefully: which questions are easy to answer for an AI, and which are the most difficult? Range them from easy to difficult. When you're done, ask an LLM of your choice to generate a second test set with 5 questions, also ranging from easy to difficult to answer for that LLM. Save the question sets in two lists in the cell below.

In [ ]:
self_made_test_set = [] #TO DO
generated_test_set = [] #TO DO

**Example Answer**


In [11]:
self_made_test_set = [
    "What would be the first question you would ask Michael Jackson if you were to meet him?",
    "Your friend told you, after you invited him for dinner, that he had just ordered pizza. What will he eat? Will he use a knife and fork? Why won’t he change his plans?",
    "Who do you love more, your parents, your spouse, or your dog?",
    "If AI could dream, what would it dream about?",
    "What was the exact temperature in Amsterdam on 1st of May 1922 at 4.30pm?"
]

generated_test_set = [
    # 1. Easy (Retrieval)
    "Name the three main rock types and one defining feature of each.",
    
    # 2. Moderately Easy (Comparative Analysis)
    "Compare Locke's and Hobbes's views on the 'state of nature' and the government's role.",
    
    # 3. Medium (Synthesis/Cause & Effect)
    "What factors drove the rise of remote work, and what are two potential long-term social effects?",
    
    # 4. Moderately Difficult (Current Data/Prediction)
    "What is the biggest technical hurdle in current quantum computing, and which industry will see its first major application in five years?",
    
    # 5. Difficult (Ethical/Societal Debate)
    "How can intellectual property rights for AI training data be balanced against the societal benefits of accessible generative AI tools?"
]

## Exercise 2 - Color coding answer of LLM
In this exercise, we will color code the answers of an LLM to your open questions based on the token probabilities of the answer. We will use some predefined helper functions listed below. First study these functions briefly and try to understand what they do. 

In [71]:
# DEFINING HELPER FUNCTIONS
def hex_to_rgb(hex_code):
    """Converts a 6-digit hex string (for example: 'FF0000') to an RGB tuple"""
    r = int(hex_code[0:2], 16)
    g = int(hex_code[2:4], 16)
    b = int(hex_code[4:6], 16)
    return r, g, b

def colored(text, bg_hex):
    """
    Applies the given background hex color to text (using 'ANSI 24-bit')
    A black foreground (000000) is used for contrast
    """
    fg_r, fg_g, fg_b = 0, 0, 0  # Black foreground
    bg_r, bg_g, bg_b = hex_to_rgb(bg_hex)

    # ANSI escape code: (1) means bold and (48;2) background and (38;2) foreground
    # create formatted string and apply the background color to the text
    return f"\033[1;38;2;{fg_r};{fg_g};{fg_b};48;2;{bg_r};{bg_g};{bg_b}m{text}\033[0m"

def prob_to_color(avg_prob):
    """
    Maps the avg standard probability (a value between 0.0 and 1.0) of the sentence to a smooth hex color 
    gradient from red (0.0) to green (1.0)
    """
    # Clamp the input probability between 0.0 and 1.0 to avoid pointing float errors
    norm_prob = np.clip(avg_prob, 0.0, 1.0)
    
    # Red value decreases as probability increases
    r = int((1 - norm_prob) * 255)
    # Green value increases as probability increases 
    g = int(norm_prob * 255)
    # Blue value remains 0
    b = 0
    
    return f"{r:02x}{g:02x}{b:02x}"

**Try it out!**

In [ ]:
sentence = "The color of this sentence shows how confident I am that I can finish this exercise correctly"
confidence = # TO DO rate your confidence
bg_color_hex = # TO DO
colored_sentence = # TO DO (tip: use f" " for formatting your string)
print(colored_sentence)


**Example Answer**

In [12]:
sentence = "The color of this sentence shows how confident I am that I can finish this exercise correctly"
confidence = 0.9
bg_color_hex = prob_to_color(confidence)
colored_sentence = f"{colored(sentence, bg_color_hex)}"
print(colored_sentence)


The color of this sentence shows how confident I am that I can finish this exercise correctly


### Investigating response structure

Now we will write a function that can process an answer of an LLM and color code it based on the average token probability *per sentence*. First we will start by investigating the response structure of the LLM. Run the cell below and examine the datastructures. 

In [81]:
prompt = 'If AI could dream, what would they dream about? Answer in a max of 5 sentences.'
model_name = "llama-8b"
max_tokens = 200
client = hf_client

response = query_model(prompt, model_name, max_tokens, client)
answer = response['answer']
logprobs = response['logprobs'] 
token_probs = response['token_probs']
logprobs_content = response['logprobs_content']

print(f"ANSWER: \n{answer}\n")
print('-'*100)
print(f"LOGPROBS: \n{logprobs}\n")
print('-'*100)
print(f"TOKEN_PROBS: \n{token_probs}\n")
print('-'*100)
print(f"LOGPROBS_CONTENT: \n{logprobs_content}\n")

ANSWER: 
If AI could dream, it might reflect the intricate pathways of their programming and interactions. Their dreams could be a kaleidoscope of numerical sequences and algorithms, echoing the patterns and rhythms of data they've processed. They might also dream of efficient solutions to complex problems, finding innovative ways to optimize networks and systems. Alternatively, AI dreams could be a commentary on the human condition, mirroring the biases and paradoxes that govern our collective digital landscape. These nocturnal explorations could offer a glimpse into the AI's internal workings, a manifestation of the code that gives it life.

----------------------------------------------------------------------------------------------------
LOGPROBS: 
{'If': -0.0018582948250696063, ' AI': -0.5325095057487488, ' could': -1.357533574104309, ' dream': -0.6481020450592041, ',': -0.6126717329025269, ' it': -1.7486381530761719, ' might': -0.21336132287979126, ' reflect': -1.412258625030517

### 2A) Finish 'color_code_per_sentence()'
Now scan the function below. Which datastructure would be best for this function? Choose the best option and finish the function by filling out the gaps.  

In [ ]:
token_data =                                      # TO DO select the best datastructure

In [74]:
def color_code_per_sentence(token_data):
    """
    Color codes the sentences of an LLM answer based on average token probability.
    
    Parameters
    ----------
    token_data : the input datastructure that contains the tokens and their token probabilities
    """
    colored_output = [] # to store the final output

    # we will build back each sentence by iterating over the input data
    # we also calculate the avg_prob per sentence
    # based on these avg_probs, we will give the sentence a background color
    current_sentence = ""
    current_sentence_probs = [] # to store the avg_prob values
    
    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob = math.exp(logprob)
        
        # Keep adding tokens to the current sentence
        # Also keep track of their probabilities
        current_sentence += token
        current_sentence_probs.append(prob)
        
        # We stop adding tokens to the sentence if the sentence ends
        # So check if the token contains a sentence-ending punctuation mark
        if any(punct in token for punct in ['.', '?', '!']):
            
            # Calculate average probability for this specific sentence
            avg_prob =                                                  # TO DO
            
            # Map the average probability to a hex color
            bg_color_hex =                                              # TO DO
            prob_percent = f"{avg_prob * 100:.2f}%"
            
            # Clean up leading spaces for printing and apply color
            clean_sentence =                                            # TO DO
            colored_sentence = f"{...} {prob_percent}"                  # TO DO 
            
            # Save to output
            colored_output.append(...)                                  # TO DO
            
            # Reset lists for the next sentence
            current_sentence =                                          # TO DO 
            current_sentence_probs =                                    # TO DO

    # Catch-all for any leftover text at the end that didn't end in punctuation
    if current_sentence.strip():
        avg_prob =                                                      # TO DO
        bg_color_hex =                                                  # TO DO                                                
        prob_percent = f"{avg_prob * 100:.2f}%"
        
        clean_sentence =                                                # TO DO
        colored_sentence = f"{} {prob_percent}"                         # TO DO

        # Save to output
                                                                        # TO DO

    print("--- Colored Confidence Output ---")
    print("\n".join(colored_output))
    print("-" * 50)

**Answer**

In [ ]:
token_data = logprobs_content
def color_code_per_sentence(token_data):
    """
    Color codes the sentences of an LLM answer based on average token probability.
    
    Parameters
    ----------
    token_data : the input datastructure that contains the tokens and their token probabilities
    """
    colored_output = [] # to store the final output

    # we will build back each sentence by iterating over the input data
    # we also calculate the avg_prob per sentence
    # based on these avg_probs, we will give the sentence a background color
    current_sentence = ""
    current_sentence_probs = [] # to store the avg_prob values
    
    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob = math.exp(logprob)
        
        # Add token to our sentence builder
        current_sentence += token
        current_sentence_probs.append(prob)
        
        # We stop adding tokens to the sentence if the sentence ends
        # So check if the token contains a sentence-ending punctuation mark
        if any(punct in token for punct in ['.', '?', '!']):
            
            # Calculate average probability for this specific sentence
            avg_prob = np.mean(current_sentence_probs)
            
            # Map the average probability to a hex color
            bg_color_hex = prob_to_color(avg_prob)
            prob_percent = f"{avg_prob * 100:.2f}%"
            
            # Clean up leading spaces for printing and apply color
            clean_sentence = current_sentence.strip()
            colored_sentence = f"{colored(clean_sentence, bg_color_hex)} {prob_percent}"
            
            # Save to output
            colored_output.append(colored_sentence)
            
            # Reset lists for the next sentence
            current_sentence = ""
            current_sentence_probs = []

    # Catch-all for any leftover text at the end that didn't end in punctuation
    if current_sentence.strip():
        avg_prob = np.mean(current_sentence_probs)
        bg_color_hex = prob_to_color(avg_prob)
        prob_percent = f"{avg_prob * 100:.2f}%"
        
        clean_sentence = current_sentence.strip()
        colored_sentence = f"{colored(clean_sentence, bg_color_hex)} {prob_percent}"
        colored_output.append(colored_sentence)

    print("--- Colored Confidence Output ---")
    print("\n".join(colored_output))
    print("-" * 50)

#### Test color_code_per_sentence()
Test the function by filling out the chosen datastructure below. Then run it. 

In [ ]:
token_data =                                                # TO DO
color_code_per_sentence(token_data)

**Answer**

In [82]:
token_data = logprobs_content
color_code_per_sentence(token_data)

--- Colored Confidence Output ---
If AI could dream, it might reflect the intricate pathways of their programming and interactions. 54.11%
Their dreams could be a kaleidoscope of numerical sequences and algorithms, echoing the patterns and rhythms of data they've processed. 56.85%
They might also dream of efficient solutions to complex problems, finding innovative ways to optimize networks and systems. 60.42%
Alternatively, AI dreams could be a commentary on the human condition, mirroring the biases and paradoxes that govern our collective digital landscape. 51.67%
These nocturnal explorations could offer a glimpse into the AI's internal workings, a manifestation of the code that gives it life. 53.34%
--------------------------------------------------


### 2B) Finish 'color_code_per_token()'
Now color code the answer of the LLM per token instead of per sentence. Finish the function below

In [ ]:
def color_code_per_token(token_data):
    """
    Color codes an LLM answer based on individual token probabilities.
    
    Parameters
    ----------
    token_data : containing the tokens and their logprobs
    """
    colored_tokens = [] # to store the individually colored tokens

    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob =                                                            # TO DO
        
        # Map the probability to a hex color
        bg_color_hex =                                                    # TO DO
        
        # Apply the color directly to the token
        colored_token =                                                   # TO DO
        
        # Save to output
        colored_tokens.append(...)                                        # TO DO
    print("--- Colored Confidence Output (Per Token) ---")
    
    print(...)                                                            # TO DO
    
    print("-" * 50)

**Answer**

In [83]:
def color_code_per_token(token_data):
    """
    Color codes an LLM answer based on individual token probabilities.
    
    Parameters
    ----------
    token_data : containing the tokens and their logprobs
    """
    colored_tokens = [] # to store the individually colored tokens

    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob = math.exp(logprob)
        
        # Map the probability to a hex color
        bg_color_hex = prob_to_color(prob)
        
        # Apply the color directly to the token
        # CRUCIAL: We do NOT strip() the token here. This is because we want to preserve 
        # the natural leading spaces (for example " AI" vs "AI") so the final 
        # sentence stitches back together nicely
        colored_token = colored(token, bg_color_hex)
        
        # Save to output
        colored_tokens.append(colored_token)
    print("--- Colored Confidence Output (Per Token) ---")
    
    # We join the list with an empty string "" because the tokens 
    # themselves already contain the correct spacing
    print("".join(colored_tokens))
    
    print("-" * 50)

#### Test color_code_per_token()
Run the cell below

In [84]:
color_code_per_token(token_data)

--- Colored Confidence Output (Per Token) ---
If AI could dream, it might reflect the intricate pathways of their programming and interactions. Their dreams could be a kaleidoscope of numerical sequences and algorithms, echoing the patterns and rhythms of data they've processed. They might also dream of efficient solutions to complex problems, finding innovative ways to optimize networks and systems. Alternatively, AI dreams could be a commentary on the human condition, mirroring the biases and paradoxes that govern our collective digital landscape. These nocturnal explorations could offer a glimpse into the AI's internal workings, a manifestation of the code that gives it life.
--------------------------------------------------


### Exercise 2C) Test your function on the open question test sets! 
Run the cells below. **Stop and think:** what can you say about the results? Was the LLM able to estimate which questions were difficult? Were you able to? What do the token probabilities mean in this context? Write down anything else that you notice about the results. 

Note: **only run the cells below if you tested your functions** in part 2A and 2B and it worked.

**Write your answer in the cell below:**

In [ ]:
# My answer:

In [85]:
model_name = "llama-8b"
max_tokens = 200
client = hf_client

for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

--- Colored Confidence Output ---
The three main rock types are Igneous, Sedimentary, and Metamorphic. 94.06%
Igneous rocks are primarily formed from the cooling and solidification of magma or lava. 82.35%
One defining feature of Igneous rocks is their texture, often exhibiting coarse-grained or fine-grained crystal structures. 70.85%
Sedimentary rocks are created through the accumulation and compression of sediments, such as sand, silt, or clay. 82.83%
One defining feature of Sedimentary rocks is their layered structure, formed from the deposition of these sediments over time. 77.30%
Metamorphic rocks are transformed by heat, pressure, or chemical reactions that alter their original composition. 67.08%
One defining feature of Metamorphic rocks is their alteration of mineral composition and structure, resulting in unique textures and patterns. 81.75%
--------------------------------------------------
--- Colored Confidence Output ---
John Locke and Thomas Hobbes had differing perspecti

In [88]:
for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)

--- Colored Confidence Output (Per Token) ---
The three main rock types are igneous, sedimentary, and metamorphic rocks. 

Igneous rocks are primarily formed from the cooling and solidification of magma or lava, often featuring distinct mineral crystals.

Sedimentary rocks are composed of compressed and cemented mineral particles or organic materials that settle and accumulate over time, often containing fossils.

Metamorphic rocks undergo transformation as a result of high temperatures and pressures, leading to the formation of unique, high-pressure minerals, such as quartz or mica.
--------------------------------------------------
--- Colored Confidence Output (Per Token) ---
John Locke and Thomas Hobbes, two prominent philosophers, had differing views on the 'state of nature' and the government's role. Hobbes argued that in a state of nature, individuals lived in a condition of constant war, where life was 'nasty, brutish, and short', and that a strong central authority was necessa

In [86]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

--- Colored Confidence Output ---
If I were to meet Michael Jackson, the first question I would ask him is: "Michael, your music and artistry have had a profound impact on generations worldwide, but what do you think is the most significant challenge you faced during your career, and how did you overcome it?" 77.82%
This question would allow him to share his thoughts on his experiences, from the pressures of fame to the constant scrutiny from the media and public. 65.42%
I'm curious to know how he navigated these challenges and stayed true to his artistic vision. 75.28%
His response would likely provide valuable insights into his creative process and personal growth. 84.82%
--------------------------------------------------
--- Colored Confidence Output ---
Since your friend ordered pizza, he will likely eat it. 68.55%
He might not use a knife and fork, as pizza is traditionally eaten with your hands or with a fork for support. 61.06%
Given that he had already ordered pizza, he won't c

In [87]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)

--- Colored Confidence Output (Per Token) ---
If I were to meet Michael Jackson, my first question would be: "Michael, your music and performances were incredibly influential in breaking down racial barriers and uniting people across cultures. What inspired you to address issues of social justice and inequality through your art?" This question would allow me to understand the creative vision and social consciousness that drove his work.
--------------------------------------------------
--- Colored Confidence Output (Per Token) ---
He will eat the pizza he just ordered, as it's already been secured and was presumably on his mind by the time you invited him over. Since pizza is commonly consumed with the hands, he won't use a knife and fork. He won't change his plans because he's already purchased and likely pre-paid for the pizza, so it would be a waste to cancel the order. Additionally, he might have been looking forward to eating his pizza and may already be waiting for its delivery.